**Morgage Rate Enrichment**

*Objective: Fetch the FRED MORTGAGE30US series, resample it from weekly to monthly frequency, and merge it onto both combined datasets using a year-month key derived from transaction dates.*

In [21]:
import pandas as pd

folder = r"C:\Users\khush\idx files"
reports_folder = r"C:\Users\khush\Desktop\IDX-Exchange\Reports"

sold = pd.read_csv(folder + r"\sold_combined.csv", low_memory=False)
listings = pd.read_csv(folder + r"\listings_combined.csv", low_memory=False)

In [22]:
#fetching morgage rate fata from FRED 

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

In [23]:
mortgage.head()

,date,rate_30yr_fixed
0,1971-04-02,7.33
1,1971-04-09,7.31
2,1971-04-16,7.31
3,1971-04-23,7.31
4,1971-04-30,7.29


In [24]:
# Step 2 – Resample weekly rates to monthly averages

mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (
 mortgage.groupby('year_month')['rate_30yr_fixed']
 .mean()
 .reset_index()
)
mortgage_monthly.head()

,year_month,rate_30yr_fixed
0,1971-04,7.3100
1,1971-05,7.4250
2,1971-06,7.5300
3,1971-07,7.6040
4,1971-08,7.6975


In [25]:
# Step 3 – Create a matching year_month key on the MLS datasets
# Sold dataset — key off CloseDate
sold['year_month'] = pd.to_datetime(sold['CloseDate']).dt.to_period('M')
# Listings dataset — key off ListingContractDate
listings['year_month'] = pd.to_datetime(
 listings['ListingContractDate']
).dt.to_period('M')

In [26]:
print(sold.head())
print(listings.head())
print(sold[['CloseDate', 'year_month']].head())

           Flooring ViewYN WaterfrontYN BasementYN PoolPrivateYN  \
0  Carpet,Tile,Wood   True          NaN        NaN         False   
1               NaN  False          NaN        NaN         False   
2               NaN  False          NaN        NaN         False   
3               NaN   True          NaN        NaN         False   
4              Tile   True          NaN        NaN         False   

   OriginalListPrice  ListingKey                  ListAgentEmail   CloseDate  \
0           499000.0   551985747           jwachter@cbnorcal.com  2024-01-26   
1           759900.0   522107581            mdarwich12@gmail.com  2024-01-05   
2           739900.0   510919001            mdarwich12@gmail.com  2024-01-05   
3          2100000.0  1067652762  jenniferarielkennedy@gmail.com  2024-01-02   
4          1950000.0  1063453216               kira@mendosir.com  2024-01-22   

   ClosePrice  ...        HighSchoolDistrict PostalCode  AssociationFee  \
0    240000.0  ...                 

In [27]:
# Step 4 – Merge
sold_with_rates = sold.merge(mortgage_monthly, on='year_month', how='left')
listings_with_rates = listings.merge(mortgage_monthly, on='year_month', how='left')

In [28]:
# Step 5 – Validate the merge
# Check for any unmatched rows (rate should not be null)
print(sold_with_rates['rate_30yr_fixed'].isnull().sum())
print(listings_with_rates['rate_30yr_fixed'].isnull().sum())

0
0


In [29]:
# Preview
print(
 sold_with_rates[
 ['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']
 ].head()
)

    CloseDate year_month  ClosePrice  rate_30yr_fixed
0  2024-01-26    2024-01    240000.0           6.6425
1  2024-01-05    2024-01    815000.0           6.6425
2  2024-01-05    2024-01    810000.0           6.6425
3  2024-01-02    2024-01   2100000.0           6.6425
4  2024-01-22    2024-01   1950000.0           6.6425


In [30]:
sold_with_rates.to_csv(reports_folder + r"\sold_with_rates.csv", index=False)
listings_with_rates.to_csv(reports_folder + r"\listings_with_rates.csv", index=False)